# Wilcoxon Significance Test — IBEX35 Direction Models

One-sample Wilcoxon signed-rank test: are per-fold `balanced_accuracy` scores
significantly greater than random chance (0.5)?

**Scope:** `expanding` CV · `discrete` target · horizons **h1** and **h5**  
**Models:** `rf`, `xgb`, `gru`, `lstm`, `cnn_gru`, `cnn_lstm`, `markov`  
**Baseline:** balanced accuracy = 0.5 (random classifier)  
**Test:** one-sided (`alternative='greater'`), α = 0.05

## 0. Config

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

from config import ARTIFACTS_PATH

ARTIFACTS_DIR = ARTIFACTS_PATH
ALPHA         = 0.05
MODELS        = ['rf', 'xgb', 'gru', 'lstm', 'cnn_gru', 'cnn_lstm', 'markov']
HORIZONS      = [1, 5]

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. Load artifacts & extract per-fold balanced accuracy

In [ ]:
# scores[horizon][model] = np.ndarray of per-fold balanced_accuracy values
scores = {h: {} for h in HORIZONS}

for h in HORIZONS:
    print(f'\n--- horizon h{h} ---')
    for model in MODELS:
        path = ARTIFACTS_DIR / f'{model}_h{h}_expanding_discrete.pkl'
        if not path.exists():
            print(f'  WARNING: {path.name} not found — skipping')
            continue
        artifact = joblib.load(path)
        cv_metrics = artifact.get('cv_metrics', [])
        fold_scores = np.array([
            fold['balanced_accuracy']
            for fold in cv_metrics
            if 'balanced_accuracy' in fold
        ])
        if len(fold_scores) == 0:
            print(f'  WARNING: {path.name} has no balanced_accuracy in cv_metrics — skipping')
            continue
        scores[h][model] = fold_scores
        print(f'  {model:12s}  n_folds={len(fold_scores):3d}  '
              f'mean={fold_scores.mean():.4f}  std={fold_scores.std():.4f}')

## 2. Wilcoxon signed-rank tests

In [ ]:
rows = []

for h in HORIZONS:
    for model, fold_scores in scores[h].items():
        n = len(fold_scores)
        mean_ba = float(fold_scores.mean())
        std_ba  = float(fold_scores.std())

        if n < 2:
            rows.append({
                'horizon': h, 'model': model, 'n_folds': n,
                'mean_ba': mean_ba, 'std_ba': std_ba,
                'W': np.nan, 'p_value': np.nan, 'significant': False,
            })
            continue

        # One-sample Wilcoxon: test whether median(scores) > 0.5
        differences = fold_scores - 0.5

        # All differences identical => test undefined
        if np.all(differences == differences[0]):
            rows.append({
                'horizon': h, 'model': model, 'n_folds': n,
                'mean_ba': mean_ba, 'std_ba': std_ba,
                'W': np.nan, 'p_value': np.nan, 'significant': False,
            })
            continue

        W, p = stats.wilcoxon(differences, alternative='greater')
        rows.append({
            'horizon': h, 'model': model, 'n_folds': n,
            'mean_ba': mean_ba, 'std_ba': std_ba,
            'W': W, 'p_value': p,
            'significant': bool(p < ALPHA),
        })

df_results = pd.DataFrame(rows)

## 3. Summary tables

In [ ]:
def _style_table(df_h):
    styled = (
        df_h[['model', 'n_folds', 'mean_ba', 'std_ba', 'W', 'p_value', 'significant']]
        .sort_values('mean_ba', ascending=False)
        .style
        .format({
            'mean_ba': '{:.4f}',
            'std_ba':  '{:.4f}',
            'W':       '{:.1f}',
            'p_value': '{:.4f}',
        })
        .apply(
            lambda row: [
                'background-color: #d4edda' if row['significant'] else ''
                for _ in row
            ],
            axis=1
        )
    )
    return styled

for h in HORIZONS:
    df_h = df_results[df_results['horizon'] == h].copy()
    if df_h.empty:
        print(f'No results for h{h}')
        continue
    print(f'\n=== Horizon h{h} ===')
    display(_style_table(df_h))

## 4. Distribution plots

In [ ]:
def _star(p):
    if pd.isna(p):   return ''
    if p < 0.001:    return '***'
    if p < 0.01:     return '**'
    if p < 0.05:     return '*'
    return 'ns'

for h in HORIZONS:
    available = [(m, s) for m, s in scores[h].items()]
    if not available:
        continue

    # Sort models by mean score descending
    row_map = df_results[df_results['horizon'] == h].set_index('model')
    available_sorted = sorted(
        available,
        key=lambda x: row_map.loc[x[0], 'mean_ba'] if x[0] in row_map.index else 0,
        reverse=True,
    )
    model_names  = [m for m, _ in available_sorted]
    model_scores = [s for _, s in available_sorted]

    fig, ax = plt.subplots(figsize=(max(8, len(model_names) * 1.4), 5))

    parts = ax.violinplot(
        model_scores,
        positions=range(len(model_names)),
        showmedians=True,
        showextrema=True,
    )
    for pc in parts['bodies']:
        pc.set_alpha(0.55)

    # Overlay individual fold points
    for i, s in enumerate(model_scores):
        jitter = np.random.default_rng(i).uniform(-0.08, 0.08, size=len(s))
        ax.scatter(np.full(len(s), i) + jitter, s, s=18, zorder=3, alpha=0.7)

    # Baseline
    ax.axhline(0.5, color='red', lw=1.2, ls='--', label='Baseline (0.5)')

    # Significance stars above each violin
    for i, name in enumerate(model_names):
        if name not in row_map.index:
            continue
        p = row_map.loc[name, 'p_value']
        top = max(model_scores[i]) + 0.012
        ax.text(i, top, _star(p), ha='center', va='bottom', fontsize=11,
                color='#155724' if (not pd.isna(p) and p < ALPHA) else '#888')

    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names, fontsize=9)
    ax.set_ylabel('Balanced Accuracy (per fold)')
    ax.set_title(f'CV fold score distributions — h{h} expanding discrete\n'
                 f'(*p<0.05  **p<0.01  ***p<0.001  ns=not significant, one-sided Wilcoxon vs 0.5)')
    ax.legend(loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.show()

## 5. Interpretation

In [ ]:
from IPython.display import Markdown, display

lines = []
for h in HORIZONS:
    lines.append(f'### Horizon h{h}')
    df_h = df_results[df_results['horizon'] == h].sort_values('mean_ba', ascending=False)
    for _, row in df_h.iterrows():
        sig_str = (
            f'**significantly better than chance** ({_star(row["p_value"])}; p={row["p_value"]:.4f})'
            if row['significant']
            else f'*not significantly different from chance* (p={row["p_value"]:.4f})'
            if not pd.isna(row['p_value'])
            else '*test not applicable (insufficient folds or zero variance)*'
        )
        lines.append(
            f'**{row["model"]} (h{h}):** '
            f'mean BA = {row["mean_ba"]:.4f} ± {row["std_ba"]:.4f}, '
            f'W = {row["W"]:.1f}, '
            f'{sig_str} '
            f'(α = {ALPHA}, one-sided Wilcoxon signed-rank, n = {int(row["n_folds"])} folds).'
        )
    lines.append('')

display(Markdown('\n\n'.join(lines)))